# EE 4XX - Lab 1

[TODO: Team member names here]

### Imports

In [2]:
import cv2
import numpy as np
import glob
import time
from sympy import symbols, cos, sin, Matrix
from pymycobot.mycobot import MyCobot

## Task 1: Camera Detection
To be able to detect and range objects properly, the distortion introduced by the camera must be known.  One model is the pinhole camera model, in which radial and tangential distortion can be calculated through a camera's intrinsic parameters.  These intrinsics allows us to undistort the image, which is necessary for being able to accurately detect and range objects.

### 1.1 - Capture calibration photos
One method for calibration is to calculate distortion based on photos of a known pattern, such as a checkerboard.  To do this, the corners lying inside the pattern are detected from the image (see https://www.mathworks.com/help/vision/ug/calibration-patterns.html for a visualization).  Since we know that the lines on the checkerboard are parallel and we know the physical dimensions of the squares on the checkerboard, we can find out how much the camera distorts the image.

In order to accurately calculate the camera intrinsics, multiple images (10 or more) should be taken, with the checkerboard appearing in different areas of the image and at different angles.  One possible way to capture the minimum of 10 photos is:
1. Checkerboard centered
2. Top-left tilted
3. Top-right tilted
4. Bottom-left tilted
5. Bottom-right tilted
6. Close distance
7. Medium distance
8. Far distance
9. Rotated left
10. Rotated right

Additionally, the physical checkerboard itself should be undistorted (e.g., attached to a flat surface, such as a clipboard) to avoid introducing additional distortion on top of the camera's intrinsic distortion. Finally, make sure the camera and the checkerboard are stationary when taking the photo to avoid motion blur.



In [3]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# TODO: How many points on each axis of the checkerboard?
dim_1 = 10
dim_2 = 7

# Prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(9,6,0)
# TODO: How many object points should be stored for a n x m checkerboard? 
#       Change the first dimension in (None, 3) to this value.
objp = np.zeros((dim_1 * dim_2, 3), np.float32)
objp[:,:2] = np.mgrid[0:dim_1, 0:dim_2].T.reshape(-1,2)

# Compensate for square size (measure physically)
square_size = 0.022
objp *= square_size
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3D point in real world space
imgpoints = [] # 2D points in image plane.

In [4]:
# TODO: Run the provided code and capture images using the camera.
# Check that the points are detected correctly before saving! 
#       To capture an image: press space (' ')
#       To save an image: press 's'
#       To retake an image: press 'r'
#       To quit: press 'q'

cap = cv2.VideoCapture(0)
img_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    display = frame.copy()
    cv2.imshow("Calibration", display)

    key = cv2.waitKey(1) & 0xFF

    if key == ord(' '):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, (dim_1, dim_2), None)

        if found:
            candidate = frame.copy()

            corners2 = cv2.cornerSubPix(
                gray, corners, (11, 11), (-1, -1), criteria
            )

            cv2.drawChessboardCorners(
                candidate, (dim_1, dim_2), corners2, found
            )

            cv2.imshow("Candidate - press s to save, r to retake", candidate)

            candidate_key = cv2.waitKey(0) & 0xFF

            if candidate_key == ord('s'):
                filename = f"images/calib_{img_idx}.jpg"
                cv2.imwrite(filename, frame)
                print(f"Saved {filename}")
                img_idx += 1

            elif candidate_key == ord('r'):
                print("Retaking image.")

            cv2.destroyWindow("Candidate - press s to save, r to retake")

        else:
            print("Checkerboard not detected, try again.")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 1.2 - Calculate and save camera intrinsics

Now that the images are captured, we need to process them and save the camera intrinsics to a file (this way calibration only has to be run once).

In [5]:
images = glob.glob('images/*.jpg')

print(len(images))

for fname in images:
    img = cv2.imread(fname)

    # Convert image to grayscale for processing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (dim_1, dim_2), None)
 
    # If found, add object points, image points (after refining them)
    # TODO: What do we append to objpoints, imgpoints?
    if ret:
        objpoints.append(objp)

        # Location of corners in image
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

# TODO: First three arguments for
#       cv2.calibrateCamera(objectPoints, imagePoints, imageSize, None, None) 
#       (last two are None)
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

# TODO: Save calibration, distortion matrices to .csv for future use
np.savetxt('calib_mtx.csv', mtx, delimiter=',')
np.savetxt('calib_dist.csv', dist, delimiter=',')

9


Check that the undistortion works (run provided code)

In [6]:
img = cv2.imread(images[0])
h, w = img.shape[:2]
newcammtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

undistort = False
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    if (undistort):
        frame = cv2.undistort(frame, mtx, dist, None, newcammtx)
        # x, y, w, h = roi
        # frame = frame[]
    
    cv2.imshow("Camera", frame)

    key = cv2.waitKey(1)

    if key == ord('u'):
        undistort = not undistort
    elif key == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()
cv2.waitKey(1)

-1

### 1.3 - Verify marker detection and ranging

In [ ]:
aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

# TODO: Measure ArUco tag size and update variable
tagSize = 38 / 1000 # m

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    rvecs = []
    tvecs = []
    
    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            rvecs.append(rvec)
            tvecs.append(tvec)
            distance = np.linalg.norm(tvec) 
            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)

            # TODO: calculate location of centroid based on dimensions of block and pose vector
            # Centroid is (block length / 2) along block's pose vectors rx, ry, rz

    cv2.imshow('ArUco', frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

-1

### Task 2: Robotic Arm Manipulation

In [ ]:
START_POS = [0,0,0,0,0,0]
NEUTRAL = [0, -40, 55, 0, 30, 0]
WATCHING_POS = [88.3, 0, 150, -180, 46.5, -180]
bot = MyCobot('/dev/ttyACM0', 115200)

try:
    mtx = np.array(np.loadtxt('calibration/calib_mtx.csv', delimiter=','))
    dist = np.array(np.loadtxt('calibration/calib_dist.csv', delimiter=','))
except FileNotFoundError:
    print('Files not found, make sure you ran calibration!')

aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

tagSize = 38 / 1000 # m

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

def trans(a, alpha, d, theta):
    return Matrix([
        [cos(theta), -sin(theta), 0, a],
        [sin(theta)*cos(alpha), cos(theta)*cos(alpha), -sin(alpha), -sin(alpha)*d],
        [sin(theta)*sin(alpha), cos(theta)*sin(alpha), cos(alpha), cos(alpha)*d],
        [0, 0, 0, 1]
    ])

In [ ]:
# new pose approach
def rvec_tvec_to_transform(rvec, tvec):
    R, _ = cv2.Rodrigues(rvec)

    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = tvec.flatten()

    return T
# x-y-z, R-P-Y to homogeneous transform
def pose_vector_to_transform(x, y, z, rx, ry, rz, degrees=True):
    """
    Converts pose vector into homogeneous transformation matrix.

    Parameters
    ----------
    x, y, z : float
        Position coordinates

    rx, ry, rz : float
        Roll, pitch, yaw angles

    degrees : bool
        True if angles are in degrees

    Returns
    -------
    T : np.ndarray
        4x4 homogeneous transform
    """

    # Convert to radians if necessary
    if degrees:
        rx = np.radians(rx)
        ry = np.radians(ry)
        rz = np.radians(rz)

    # Rotation about X-axis (roll)
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)]
    ])

    # Rotation about Y-axis (pitch)
    Ry = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])

    # Rotation about Z-axis (yaw)
    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0, 0, 1]
    ])

    # Combined rotation matrix
    # Convention: R = Rz * Ry * Rx
    R = Rz @ Ry @ Rx

    # Construct homogeneous transform
    T = np.eye(4)

    T[:3, :3] = R
    T[:3, 3] = [x, y, z]

    return T

def transform_to_pose(T, degrees=True):
    R = T[:3, :3]
    t = T[:3, 3]

    # ZYX convention: R = Rz(yaw) @ Ry(pitch) @ Rx(roll)
    roll = np.arctan2(R[2, 1], R[2, 2])
    pitch = np.arctan2(-R[2, 0], np.sqrt(R[2, 1]**2 + R[2, 2]**2))
    yaw = np.arctan2(R[1, 0], R[0, 0])

    if degrees:
        roll, pitch, yaw = np.degrees([roll, pitch, yaw])

    pose = np.concatenate((t, [roll, pitch, yaw % 90]))
    return pose

In [ ]:
# TODO: Transformation matrix for camera in end-effector frame 
T_E_C = np.array([[0, 1, 0, 0],
                  [-1, 0, 0, 43],
                  [0, 0, 1, -25],
                  [0, 0, 0, 1]])

# TODO: offset from end of joint to gripper
offset = [0, 0, 40, 0, 85, 0]

In [ ]:
cv2.destroyAllWindows()
cv2.waitKey(1)

cap = cv2.VideoCapture(1)


while True:
    ret, frame = cap.read()
    if (not ret): continue

    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    key = cv2.waitKey(1)

    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            distance = np.linalg.norm(tvec) 
            t_rounded = np.round(tvec * 1000, 2).flatten().tolist()
            text = ", ".join(map(str, t_rounded))

            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.putText(frame, text, (int(square[0][0][0]), int(square[0][0][1]) + 20), cv2.FONT_HERSHEY_COMPLEX, 0.5, (255,255,255), 2)

            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)
    

    if key == ord(' '):
        if corners is not None and ids is not None:
            for square, id in zip(corners, ids):
                drawSquare(frame, square)
                ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
                distance = np.linalg.norm(tvec) 
                t_rounded = np.round(tvec * 1000, 2).flatten().tolist()
                text = ", ".join(map(str, t_rounded))

                cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
                cv2.putText(frame, text, (int(square[0][0][0]), int(square[0][0][1]) + 20), cv2.FONT_HERSHEY_COMPLEX, 1, (255,255,255), 2)
                
                T_B_E = pose_vector_to_transform(*bot.get_coords())
                T_C_block = rvec_tvec_to_transform(rvec, tvec * 1000)

                T_B_block = T_B_E @ T_E_C @ T_C_block

                pose = transform_to_pose(T_B_block)
                pose[3:4] = 0

                desired_pos = pose + offset

                print("object pos:", pose)
                print("ee pos:", bot.get_coords())
                print("desired coords:", desired_pos)

                # Move EE above desired position
                # bot.send_coords(desired_pos + [0, 0, 20, 0, 0, 0], 40)
                # time.sleep(2)

                # Move EE to desired position
                bot.send_coords(desired_pos, 40)
                time.sleep(2)

                # Grab block
                bot.set_gripper_value(70, 40)
                time.sleep(1)

                # Move to starting position
                bot.send_angles(START_POS, 40)
                time.sleep(1)

                # Drop block
                bot.set_gripper_value(100, 40, 1)
                
        else:
            print('No tags detected!')
 
    cv2.imshow('ArUco', frame)

    if key == ord('r'):
        bot.set_gripper_value(100, 40, 1)
        time.sleep(2)
        bot.send_angles(NEUTRAL, 40)
        time.sleep(2)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1) 